# Semantic Kernel 工具使用示例

## 导入所需包

In [2]:
import json
import os

from dotenv import load_dotenv

from IPython.display import display, HTML

from typing import Annotated
from openai import AsyncOpenAI

from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import FunctionCallContent, FunctionResultContent, StreamingTextContent
from semantic_kernel.functions import kernel_function

## 创建插件
Semantic Kernel 使用插件作为代理可以调用的工具。一个插件可以包含多个 `kernel_functions` 作为一个组。

在下面的示例中，我们创建一个 `DestinationsPlugin` 插件，它有两个函数：
1. 使用 `get_destinations` 函数提供目的地列表
2. 使用 `get_availability` 函数提供每个目的地的可用性

In [3]:
# 为示例定义一个示例插件
class DestinationsPlugin:
    """度假目的地列表。"""

    @kernel_function(description="提供度假目的地列表。")
    def get_destinations(self) -> Annotated[str, "返回度假目的地。"]:
        return """
        Barcelona, Spain
        Paris, France
        Berlin, Germany
        Tokyo, Japan
        New York, USA
        """

    @kernel_function(description="提供目的地的可用性。")
    def get_availability(
        self, destination: Annotated[str, "要检查可用性的目的地。"]
    ) -> Annotated[str, "返回目的地的可用性。"]:
        return """
        Barcelona - Unavailable
        Paris - Available
        Berlin - Available
        Tokyo - Unavailable
        New York - Available
        """

## 创建客户端

在本示例中，我们将使用 [GitHub Models](https://aka.ms/ai-agents-beginners/github-models) 访问 LLM。

`ai_model_id` 被定义为 `gpt-4o-mini`。尝试将模型更改为 GitHub Models 市场上可用的其他模型，以查看不同的结果。

为了使用 GitHub Models 的 `Azure Inference SDK`（用于 `base_url`），我们将在 Semantic Kernel 中使用 `OpenAIChatCompletion` 连接器。Semantic Kernel 还提供了其他 [可用连接器](https://learn.microsoft.com/semantic-kernel/concepts/ai-services/chat-completion) 用于其他模型提供商。

In [4]:
load_dotenv()
client = AsyncOpenAI(
    api_key=os.getenv("API_KEY"),
    base_url=os.getenv("API_URL"),
)

chat_completion_service = OpenAIChatCompletion(
    ai_model_id=os.getenv("MODEL_FREE_8B"),
    async_client=client,
)

## 创建代理
现在我们将通过设置代理名称和指令来创建代理。

您可以更改这些设置，以查看代理响应的差异。

In [5]:
# 创建代理
agent = ChatCompletionAgent(
    service=chat_completion_service,
    name="TravelAgent",
    instructions="回答有关旅行目的地及其可用性的问题。",
    plugins=[DestinationsPlugin()],
)

## 运行代理

现在我们将运行 AI 代理。在这个代码片段中，我们可以向 `user_input` 添加两条消息，以展示代理如何响应后续问题。

代理应该调用正确的函数来获取可用目的地列表并确认某个位置的可用性。

您可以更改 `user_inputs` 以查看代理的响应。

In [6]:
user_inputs = [
    "有哪些可用的目的地？",
    "巴塞罗那是否可用？",
    "有没有不在欧洲的可用度假目的地？",
]

async def main():
    thread: ChatHistoryAgentThread | None = None

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        agent_name = None
        full_response: list[str] = []
        function_calls: list[str] = []
        function_calls_made = []  # 跟踪检测到的函数调用

        # 用于重构流式函数调用的缓冲区
        current_function_name = None
        argument_buffer = ""

        async for response in agent.invoke_stream(
            messages=user_input,
            thread=thread,
        ):
            thread = response.thread
            agent_name = response.name
            content_items = list(response.items)

            for item in content_items:
                if isinstance(item, FunctionCallContent):
                    if item.function_name:
                        current_function_name = item.function_name

                    # 累积参数（以块形式流式传输）
                    if isinstance(item.arguments, str):
                        argument_buffer += item.arguments
                        
                    # 对于当前的 semantic-kernel 版本，立即完成函数调用
                    # 因为 FunctionResultContent 可能不会流式传输
                    if current_function_name and argument_buffer:
                        formatted_args = argument_buffer.strip()
                        try:
                            parsed_args = json.loads(formatted_args)
                            formatted_args = json.dumps(parsed_args)
                        except Exception:
                            pass  # 保持为原始字符串

                        function_calls_made.append({
                            'name': current_function_name,
                            'args': formatted_args
                        })
                        
                elif isinstance(item, FunctionResultContent):
                    # 这处理如果 FunctionResultContent 可用的情况（较新版本）
                    # 在显示结果之前完成任何待处理的函数调用
                    if current_function_name:
                        formatted_args = argument_buffer.strip()
                        try:
                            parsed_args = json.loads(formatted_args)
                            formatted_args = json.dumps(parsed_args)
                        except Exception:
                            pass  # 保持为原始字符串

                        function_calls.append(f"调用函数: {current_function_name}({formatted_args})")
                        current_function_name = None
                        argument_buffer = ""

                    function_calls.append(f"\n函数结果:\n\n{item.result}")
                elif isinstance(item, StreamingTextContent) and item.text:
                    full_response.append(item.text)

        # 如果我们检测到函数调用但在流式传输中没有获得 FunctionResultContent，
        # 我们可以推断函数被成功调用，因为我们得到了响应
        if function_calls_made and not function_calls:
            for func_call in function_calls_made:
                function_calls.append(f"调用函数: {func_call['name']}({func_call['args']})")
            
            # 由于我们知道函数被调用并且我们有响应，
            # 我们可以表明函数结果已被处理
            if len(function_calls_made) > 0:
                function_calls.append("\n函数结果已成功处理（结果用于生成上面的响应）")

        if function_calls:
            html_output += (
                "<div style='margin-bottom:10px'>"
                "<details>"
                "<summary style='cursor:pointer; font-weight:bold; color:#0066cc;'>函数调用（点击展开）</summary>"
                "<div style='margin:10px; padding:10px; background-color:#f8f8f8; "
                "border:1px solid #ddd; border-radius:4px; white-space:pre-wrap; font-size:14px; color:#333;'>"
                f"{chr(10).join(function_calls)}"
                "</div></details></div>"
            )

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>{agent_name or 'Assistant'}:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))

await main()